# 05 · Combining GNSS and InSAR

So far every inversion has used a single dataset of GNSS stations. Real studies
almost always combine measurements of different kinds, because each kind is
strong where the others are weak. This chapter introduces radar (InSAR) data,
shows how it differs from GNSS, and joins the two into one inversion, all while
keeping each dataset's influence visible and honest.

**Learning objectives**

By the end of the chapter, you will be able to:

- describe how GNSS and InSAR sample surface motion differently
- write the line-of-sight projection that turns 3-D motion into one radar number
- build a GeoDef `InSAR` dataset from a look vector
- run single-dataset and joint inversions and compare them
- read per-dataset diagnostics to check that neither input is being ignored

**Prerequisites:** Chapter 04
**Estimated time:** 60–90 minutes

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. Two instruments, two blind spots

GNSS and InSAR measure the same ground motion in almost opposite ways.

|              | GNSS                        | InSAR                          |
| ---          | ---                         | ---                            |
| where        | sparse (a few stations)     | dense (thousands of pixels)    |
| what         | full 3-D: East, North, Up   | one direction: line of sight   |
| reference    | absolute positions          | relative between pixels        |

A GNSS station reports all three components of motion, but there are rarely
many of them. InSAR blankets the ground with pixels, but each pixel measures
only a single number: the motion projected onto the direction between the
ground and the satellite. Where GNSS is spatially sparse, InSAR is dense; where
InSAR sees only one direction, GNSS sees all three. Combining them can pin down
slip that either alone would leave ambiguous.

## 2. The line-of-sight projection

A radar satellite cannot measure sideways motion that does not change its
distance to the ground. It records only the part of the displacement pointing
along its viewing direction, the **line of sight** (LOS).

Write the 3-D surface displacement as $\mathbf{u} = (u_e, u_n, u_u)$ and the
unit vector pointing from ground to satellite as
$\hat{\mathbf{l}} = (l_e, l_n, l_u)$, called the **look vector**. The measured
LOS displacement is their dot product,

$$ d_{\text{LOS}} = \mathbf{u} \cdot \hat{\mathbf{l}}
   = l_e\,u_e + l_n\,u_n + l_u\,u_u. $$

This is exactly the "combine three numbers into one" operation from Chapter 00.
A typical ascending-track satellite looks mostly downward and slightly to the
side, so its look vector is dominated by the Up component. That is why InSAR is
often described, loosely, as sensitive mostly to vertical motion, but the
horizontal terms never fully vanish, and ignoring them causes sign errors.

## 3. Rebuild the shared GNSS scenario

We start from the same megathrust, true slip, GNSS network, and 1 cm noise
used since Chapter 03. The next cells rebuild it; see Chapter 03 for the
step-by-step narration.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

%matplotlib inline

rng = np.random.default_rng(0)

In [ ]:
# The chapter 03 megathrust and its true smooth thrust asperity.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,
    strike=315.0, dip=15.0,
    length=180e3, width=90e3,
    n_length=12, n_width=6,
)
N = fault.n_patches
n_length, n_width = fault.grid_shape
along = np.arange(N) % n_length
down = np.arange(N) // n_length
bump = np.exp(-(((along - (n_length - 1) / 2) / 3.0) ** 2
                + ((down - n_width / 2) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

In [ ]:
# An 8 x 8 GNSS grid with 1 cm of seeded noise on every component.
grid_lon, grid_lat = np.meshgrid(np.linspace(98.5, 101.5, 8),
                                 np.linspace(-3.6, -0.4, 8))
station_lon, station_lat = grid_lon.ravel(), grid_lat.ravel()
n_stations = station_lon.size

G_gnss = fault.greens_matrix(station_lat, station_lon)
sigma_gnss = 0.01
d_gnss = G_gnss @ slip_true + rng.normal(0.0, sigma_gnss, 3 * n_stations)
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=d_gnss[0::3], north=d_gnss[1::3], up=d_gnss[2::3],
    sigma_east=np.full(n_stations, sigma_gnss),
    sigma_north=np.full(n_stations, sigma_gnss),
    sigma_up=np.full(n_stations, sigma_gnss),
)
print(f"{n_stations} GNSS stations")

## 4. Build a synthetic InSAR track

Now the new part. We lay a dense 20 by 20 grid of radar pixels over the same
fault, then generate their measurements in a way that makes the projection
explicit, so nothing is hidden inside a library call.

First, the pixel coordinates and a look vector. We use a single look direction
for the whole scene, typical of an ascending track: mostly up, slightly east
and north.

In [ ]:
pix_lon, pix_lat = np.meshgrid(np.linspace(99.5, 100.5, 20),
                               np.linspace(-0.4, 0.4, 20))
pix_lon, pix_lat = pix_lon.ravel(), pix_lat.ravel()
n_pixels = pix_lon.size

look_e = np.full(n_pixels, -0.38)   # slightly west-pointing
look_n = np.full(n_pixels, 0.09)    # slightly north-pointing
look_u = np.full(n_pixels, 0.92)    # mostly upward

To generate the true LOS signal, we forward-model the true slip into full 3-D
displacement at each pixel, then apply the dot product of Section 2 by hand.
This is the projection GeoDef will later apply for us; doing it once
explicitly shows there is no magic.

In [ ]:
ue, un, uu = fault.displacement(pix_lat, pix_lon,
                                slip_strike=slip_true[:N],
                                slip_dip=slip_true[N:])
los_clean = look_e * ue + look_n * un + look_u * uu     # the LOS dot product

Finally add noise. InSAR is often more precise per pixel than GNSS, so we use
a 5 mm standard deviation. (This treats each pixel's noise as independent;
Chapter 06 takes up the more realistic case where nearby pixels share
correlated errors.)

In [ ]:
sigma_los = 0.005
los = los_clean + rng.normal(0.0, sigma_los, n_pixels)
insar = geodef.data.insar(
    lon=pix_lon, lat=pix_lat, los=los, sigma=np.full(n_pixels, sigma_los),
    look_e=look_e, look_n=look_n, look_u=look_u,
)
print(f"{n_pixels} InSAR pixels")

## 5. The two views of the same slip

Side by side, the contrast is clear. The GNSS panel is a handful of arrows; the
InSAR panel is a dense, continuous image of one-directional motion.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
geodef.plot.vectors(gnss, fault, ax=ax1, title="GNSS displacements (sparse, 3-D)")
geodef.plot.insar(insar, fault, ax=ax2, title="InSAR line of sight (dense, 1-D)")
plt.tight_layout()

## 6. How GeoDef stacks the two problems

A joint inversion does not do anything conceptually new. It simply stacks the
two forward problems, one on top of the other, into one taller system:

$$ \begin{bmatrix} G_{\text{GNSS}} \\ G_{\text{InSAR}} \end{bmatrix} \mathbf{m}
   = \begin{bmatrix} \mathbf{d}_{\text{GNSS}} \\ \mathbf{d}_{\text{InSAR}} \end{bmatrix}. $$

Each dataset contributes its own rows. The GNSS rows are the interleaved
East/North/Up responses of Chapter 02; the InSAR rows are those same responses
already collapsed through the look-vector dot product, one row per pixel. Each
dataset also brings its own uncertainties, so the more precise measurements
automatically carry more weight (Chapter 03). In GeoDef you never assemble this
stack yourself: you pass a *list* of datasets to `geodef.solve`.

## 7. GNSS alone, InSAR alone, and both together

Let us invert three ways at the same smoothing strength: GNSS only, InSAR only,
and the two jointly. Passing a list to `geodef.solve` is the only change needed
for the joint case.

In [ ]:
settings = dict(components="dip", regularization="laplacian",
                regularization_strength=1.0)
result_gnss = geodef.solve(fault, gnss, **settings)
result_insar = geodef.solve(fault, insar, **settings)
result_joint = geodef.solve(fault, [gnss, insar], **settings)

Compare the recovered slip on one shared color scale.

In [ ]:
truth = slip_true[N:]
vmax = truth.max()
panels = [("True", truth), ("GNSS only", result_gnss.dip_slip),
          ("InSAR only", result_insar.dip_slip),
          ("Joint", result_joint.dip_slip)]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), constrained_layout=True)
for ax, (name, slip) in zip(axes, panels):
    geodef.plot.slip(fault, slip, ax=ax, cmap="RdBu_r",
                     vmin=-vmax, vmax=vmax, colorbar=False, title=name)
fig.colorbar(axes[0].collections[-1], ax=axes, shrink=0.8, label="Slip (m)")

The GNSS-only and joint solutions both track the truth well. InSAR alone is
badly under-constrained: a single look direction over a limited swath cannot
separate the three components of motion, so its recovered slip is the worst of
the three. Subtracting the truth makes the gap unmistakable.

In [ ]:
diffs = [(name, slip - truth) for name, slip in panels[1:]]
dmax = max(np.abs(diff).max() for _, diff in diffs)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), constrained_layout=True)
for ax, (name, diff) in zip(axes, diffs):
    geodef.plot.slip(fault, diff, ax=ax, cmap="RdBu_r",
                     vmin=-dmax, vmax=dmax, colorbar=False,
                     title=f"{name} - true")
fig.colorbar(axes[0].collections[-1], ax=axes, shrink=0.8,
             label="Recovered - true (m)")

The lesson is not that InSAR is a poor dataset. In a real study the InSAR swath
would be wider and combined with descending-track data for a second look
direction. The lesson is that a single line of sight is one number per pixel,
and one number cannot resolve three components of motion on its own.

## 8. Checking that neither dataset is ignored

When two datasets of very different size are joined, it is worth confirming
that the dense one has not simply overwhelmed the sparse one. Because we gave
each dataset a name, `geodef.invert.diagnostics` can report the fit to each
one separately.

In [ ]:
for name, diag in geodef.invert.diagnostics(result_joint).items():
    print(f"{name:6s}: reduced chi^2 = {diag.reduced_chi2:.2f}  "
          f"({diag.n_obs} obs, RMS residual {diag.rms * 1000:.1f} mm)")

Both reduced chi-squared values sit near 1, so the joint model explains each
dataset to within its stated noise; neither is being sacrificed to fit the
other. If one value were far above 1 while the other stayed near 1, it would
warn that the joint fit was favoring one dataset, or that one dataset carried a
systematic error the model cannot match. Chapter 06 revisits exactly this kind
of imbalance when InSAR noise turns out to be correlated.

**Checkpoint.** Suppose you accidentally flipped the sign of every look-vector
component. Would the recovered slip change? Would the per-dataset chi-squared?

<details><summary>Answer</summary>

Flipping the whole look vector flips the sign of every modeled LOS value. If
you flip it consistently in both the synthetic data and the dataset used for
inversion, the two sign flips cancel and nothing changes. If you flip it in
only one place, the modeled and observed LOS disagree everywhere, the InSAR
chi-squared explodes, and the recovered slip is corrupted. This is why a known
upward motion is a useful sign check on your look-vector convention.

</details>

## Checkpoint questions

**Why is InSAR not simply a vertical-displacement dataset?**

<details><summary>Answer</summary>

Its measurement mixes East, North, and Up through the look vector. The upward
term usually dominates, but the horizontal terms are always present, and
treating LOS as pure vertical introduces errors wherever horizontal motion is
significant.

</details>

**If you doubled every uncertainty in one dataset, what happens to its
influence on the joint solution?**

<details><summary>Answer</summary>

Weight is one over variance, and doubling the standard deviation quadruples the
variance, so the dataset's weight drops to one quarter. It influences the joint
solution four times less strongly.

</details>

**Why give each dataset a name?**

<details><summary>Answer</summary>

The name lets GeoDef keep the datasets separate through predictions,
residuals, and diagnostics, so you can audit each one's fit without manually
slicing rows out of a stacked vector.

</details>

## Common mistakes

- **An inconsistent look vector.** Reverse it in only one place and the modeled
  and observed LOS fight each other. Project a known displacement (a pure
  uplift, say) to check the sign before trusting a track.
- **Mixing velocities and displacements.** Combining a velocity dataset with a
  displacement dataset in one solve produces a model with meaningless units.
  Keep the physical quantities consistent.
- **Balancing by row count instead of covariance.** Dense InSAR has thousands
  of rows and sparse GNSS only dozens. It is the *uncertainties*, not the row
  counts, that should set the balance; trust the covariance weighting rather
  than trying to equalize the number of observations.

## Recap

- GNSS and InSAR sample complementary parts of the same motion: 3-D and sparse
  versus 1-D and dense.
- An InSAR pixel measures the look-vector dot product of the surface
  displacement, one number per pixel.
- A joint inversion stacks the two forward problems and weights each by its own
  covariance; in GeoDef you just pass a list of datasets.
- Per-dataset diagnostics confirm that a joint fit honors both inputs.

Chapter 06 takes the InSAR side further, replacing the assumption of
independent pixel noise with realistic spatial correlation.

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/05_multiple_datasets_solutions.ipynb`.

1. **Reweighting.** Inflate the InSAR uncertainty by factors of two and ten and
   watch the joint solution move from InSAR-leaning toward GNSS-leaning.
2. **A second look direction.** Add a descending-track InSAR dataset (a look
   vector pointing the other way in East) and quantify how much the joint
   uncertainty improves over a single track.
3. **A sign error.** Reverse one track's look vector in the inversion only, and
   describe the residual pattern and the per-dataset chi-squared it produces.
4. **Challenge: stack it yourself.** Reproduce the joint solution by explicitly
   stacking the two Green's matrices and observation vectors, and confirm you
   match `geodef.solve` on the list of datasets.

## Further reading

- Hanssen, R. F. (2001), *Radar Interferometry: Data Interpretation and Error
  Analysis*, on InSAR measurement and its error sources.
- Segall, P. (2010), *Earthquake and Volcano Deformation*, on joint geodetic
  inverse problems.
- Aster, R., Borchers, B., and Thurber, C. (2018), *Parameter Estimation and
  Inverse Problems*, 3rd ed., on weighted combination of datasets.
- [GeoDef data documentation](../docs/data.md) for the `GNSS` and `InSAR`
  constructors and their look-vector convention.
- The next course chapter is `06_correlated_noise.ipynb`.